## Аналіз A/B-тестів

Ви - аналітик даних в ІТ-компанії і до вас надійшла задача проаналізувати дані A/B тесту в популярній [грі Cookie Cats](https://www.facebook.com/cookiecatsgame). Це - гра-головоломка в стилі «з’єднай три», де гравець повинен з’єднати плитки одного кольору, щоб очистити дошку та виграти рівень. На дошці також зображені співаючі котики :)

Під час проходження гри гравці стикаються з воротами, які змушують їх чекати деякий час, перш ніж вони зможуть прогресувати або зробити покупку в додатку.

У цьому блоці завдань ми проаналізуємо результати A/B тесту, коли перші ворота в Cookie Cats було переміщено з рівня 30 на рівень 40. Зокрема, ми хочемо зрозуміти, як це вплинуло на утримання (retention) гравців. Тобто хочемо зрозуміти, чи переміщення воріт на 10 рівнів пізніше якимось чином вплинуло на те, що користувачі перестають грати в гру раніше чи пізніше з точки зору кількості їх днів з моменту встановлення гри.

Будемо працювати з даними з файлу `cookie_cats.csv`. Колонки в даних наступні:

- `userid` - унікальний номер, який ідентифікує кожного гравця.
- `version` - чи потрапив гравець в контрольну групу (gate_30 - ворота на 30 рівні) чи тестову групу (gate_40 - ворота на 40 рівні).
- `sum_gamerounds` - кількість ігрових раундів, зіграних гравцем протягом першого тижня після встановлення
- `retention_1` - чи через 1 день після встановлення гравець повернувся і почав грати?
- `retention_7` - чи через 7 днів після встановлення гравець повернувся і почав грати?

Коли гравець встановлював гру, його випадковим чином призначали до групи gate_30 або gate_40.

1. Для початку, уявімо, що ми тільки плануємо проведення зазначеного А/B-тесту і хочемо зрозуміти, дані про скількох користувачів нам треба зібрати, аби досягнути відчутного ефекту. Відчутним ефектом ми вважатимемо збільшення утримання на 1% після внесення зміни. Обчисліть, скільки користувачів сумарно нам треба аби досягнути такого ефекту, якщо продакт менеджер нам повідомив, що базове утримання є 19%.

In [24]:
import numpy as np
import pandas as pd
import scipy.stats as stats
import statsmodels.stats.api as sms
from math import ceil
from statsmodels.stats.proportion import proportion_effectsize, proportions_ztest, proportion_confint

Кількість користувачів, яку треба зібрати у кожній групі, визначається аналізом потужності. Вона залежить від:

* статистичної потужності (зазвичай 0.8),
* рівня значущості ($\alpha = 0.05$),
* очікуваної величини ефекту.

Припустимо, що базовий рівень утримання складає 19%, а після змін ми очікуємо побачити 20%. Тобто величина ефекту — 1 відсоткових пункти.

In [21]:
p1 = 0.19
p2 = 0.20
alpha = 0.05
power = 0.80

effect = proportion_effectsize(p2, p1)
analysis = sms.NormalIndPower()
n_per_group = analysis.solve_power(effect_size=effect, power=power, alpha=alpha, ratio=1.0)

print(f"Необхідно користувачів на групу: {int(ceil(n_per_group))}")
print(f"Необхідно користувачів сумарно: {int(ceil(2*n_per_group))}")

Необхідно користувачів на групу: 24638
Необхідно користувачів сумарно: 49276


Нам буде потрібно **мінімум 24638 спостережень для кожної групи**.

2. Зчитайте дані АВ тесту у змінну `df` та виведіть середнє значення показника показник `retention_7` (утримання на 7 день) по версіям гри. Сформулюйте гіпотезу: яка версія дає краще утримання через 7 днів після встановлення гри?

In [22]:
df = pd.read_csv(r"C:\Users\User\Desktop\DATA LOVES ACADEMY\HOMETASKS\statistical_hypothesis\cookie_cats.csv")

ret7_means = df.groupby("version")["retention_7"].mean()
print("Середнє retention_7 по версіях:")
print(ret7_means)

Середнє retention_7 по версіях:
version
gate_30    0.190201
gate_40    0.182000
Name: retention_7, dtype: float64


За середнім значенням **retention_7** видно, що утримання у **gate_30** трохи вище, ніж у **gate_40**.  
Тобто здається, що перенесення воріт на 40 рівень не покращило рівень утримання на 7-й день.



3. Перевірте з допомогою пасуючого варіанту z-тесту, чи дає якась з версій гри кращий показник `retention_7` на рівні значущості 0.05. Обчисліть також довірчі інтервали для варіантів до переміщення воріт і після. Виведіть результат у форматі:

    ```
    z statistic: ...
    p-value: ...
    Довірчий інтервал 95% для групи control: [..., ...]
    Довірчий інтервал 95% для групи treatment: [..., ...]
    ```

    де замість `...` - обчислені значення.
    
    В якості висновку дайте відповідь на два питання:  

      1. Чи є статистична значущою різниця між поведінкою користувачів у різних версіях гри?   
      2. Чи перетинаються довірчі інтервали утримання користувачів з різних версій гри? Про що це каже?  


In [34]:
control = df[df["version"] == "gate_30"]["retention_7"]
treatment = df[df["version"] == "gate_40"]["retention_7"]

x_control = int(control.sum()); n_control = control.shape[0]
x_treat = int(treatment.sum()); n_treat = treatment.shape[0]

count = np.array([x_control, x_treat])
nobs = np.array([n_control, n_treat])

z_stat, p_value = proportions_ztest(count=count, nobs=nobs)

ci_control = proportion_confint(count=x_control, nobs=n_control, alpha=0.05)
ci_treat = proportion_confint(count=x_treat, nobs=n_treat, alpha=0.05)

print(f"z statistic: {z_stat:.4f}")
print(f"p-value: {p_value:.4f}")
print(f"Довірчий інтервал 95% для групи control: [{ci_control[0]:.4f}, {ci_control[1]:.4f}]")
print(f"Довірчий інтервал 95% для групи treatment: [{ci_treat[0]:.4f}, {ci_treat[1]:.4f}]")

z statistic: 3.1644
p-value: 0.0016
Довірчий інтервал 95% для групи control: [0.1866, 0.1938]
Довірчий інтервал 95% для групи treatment: [0.1785, 0.1855]


      1. Чи є статистична значущою різниця між поведінкою користувачів у різних версіях гри?  

Так, різниця є статистично значущою. Утримання на 7-й день нижче в gate_40.


      2. Чи перетинаються довірчі інтервали утримання користувачів з різних версій гри? Про що це каже?  

Вони не перетинаються. Це каже про те, що утримання в gate_30 стабільно вище, ніж у gate_40

4. Виконайте тест Хі-квадрат на рівні значущості 5% аби визначити, чи є залежність між версією гри та утриманням гравця на 7ий день після реєстрації.

    - Напишіть, як для цього тесту будуть сформульовані гіпотези.
    - Проведіть обчислення, виведіть p-значення і напишіть висновок за результатами тесту.


H0: version і retention_7 незалежні (зв'язку немає)

H1: є залежність між version і retention_7

In [36]:
cont_table = pd.crosstab(df["version"], df["retention_7"])
display(cont_table)

chi2, p_value, dof, expected = stats.chi2_contingency(cont_table)

print(f"chi2 statistic: {chi2:.4f}")
print(f"p-value: {p_value:.4f}")
print(f"degrees of freedom: {dof}")

retention_7,False,True
version,,
gate_30,36198,8502
gate_40,37210,8279


chi2 statistic: 9.9591
p-value: 0.0016
degrees of freedom: 1


Відхиляємо нульову гіпотезу про незалежність версії гри та утримання на 7-й день, оскільки p-value = 0.0016 < 0.05